# TerraScout AI — Producto 2: Valuación Hedónica, Métricas e Inferencia ML

Notebook técnico completo y reproducible alineado con el repositorio `Capstone/`. Compatible con **Google Colab** (clona el repositorio automáticamente e incluye fallback de versión `scikit-learn`) y **Jupyter local**.

> **Nota para Google Colab:** si el repositorio en GitHub está público, esta notebook lo clona automáticamente en `/content/Capstone`. Si está privado, ejecutá primero `git clone` con tu token o montá la carpeta desde Google Drive.

## 1. Objetivo de negocio y definición del Target

- **Problema de negocio:** Detectar automáticamente terrenos subvaluados (oportunidades de arbitraje inmobiliario `>= 20%` respecto de su valor justo de mercado) aptos para desarrollar campings o glampings en los valles de **Calamuchita y Punilla (Córdoba)**, filtrando terrenos inviables por pendiente (`> 12°`) o riesgo ambiental (`ALTO`).
- **Fuente de datos limpia:** Consume [`data/processed/dataset_limpio.csv`](../data/processed/dataset_limpio.csv) (`894` lotes reales únicos generados por `APIFY ETL`), sin duplicar código de limpieza ni inventar datos (`ZERO DUMMY DATA`).
- **Target elegido — `y = log(precio_m2_usd)`:**
  - Los precios inmobiliarios por `m²` presentan fuerte asimetría positiva (cola derecha larga por lotes chicos o premium sobre lago/ruta).
  - Predecir `log(precio_m2_usd)` estabiliza la varianza, evita que unos pocos outliers dominen el gradiente/error cuadrático y penaliza errores en escala relativa (porcentual).
  - Para obtener el valor justo estimado en dólares se aplica la transformación inversa `precio_modelo_m2 = exp(y_hat)`.

## 2. Carga de datos limpios y reglas desacopladas

Todos los umbrales de negocio (`tipo_cambio_ars_usd`, `pendiente_alerta_grados = 12°`, `umbral_descuento_minimo_pct = 20%`, `umbral_gap_sospechoso_pct = 50%`) y los parámetros del modelo viven en [`config/business_rules.json`](../config/business_rules.json).

La arquitectura separa estrictamente:
- **Entrenamiento offline:** [`src/training.py`](../src/training.py) y [`src/run_producto2_pipeline.py`](../src/run_producto2_pipeline.py).
- **Actor Apify (100% solo inferencia):** [`actors/apify_ml/my_actor/preprocessing.py`](../actors/apify_ml/my_actor/preprocessing.py) y [`actors/apify_ml/my_actor/tooling_business.py`](../actors/apify_ml/my_actor/tooling_business.py).

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    colab_repo = Path('/content/Capstone')
    if not (colab_repo / 'config' / 'business_rules.json').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/JonathanHiriart/Capstone.git', str(colab_repo)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl', 'pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib'])
    os.chdir(colab_repo)
    REPO_ROOT = colab_repo
else:
    REPO_ROOT = Path('.').resolve()
    if not (REPO_ROOT / 'config' / 'business_rules.json').exists() and (REPO_ROOT.parent / 'config' / 'business_rules.json').exists():
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from actors.apify_ml.my_actor.preprocessing import (
    enrich_ml_features,
    load_business_rules,
    load_clean_dataset,
)
from actors.apify_ml.my_actor.tooling_business import (
    execute_ml_actor,
    format_telegram_summary,
    load_model_bundle,
    score_opportunities,
)
from src.run_producto2_pipeline import run_producto2_train
from src.training import train_and_evaluate_hedonic_model

rules = load_business_rules(str(REPO_ROOT / 'config' / 'business_rules.json'))
df_clean = load_clean_dataset(str(REPO_ROOT / 'data' / 'processed' / 'dataset_limpio.csv'))

print(f'Entorno: {"Google Colab" if IN_COLAB else "Local"} | Dataset limpio oficial: {len(df_clean)} lotes x {df_clean.shape[1]} columnas')
print(f'Valles presentes: {df_clean["valle_corredor"].value_counts().to_dict()}')
display(df_clean[['precio_usd', 'superficie_m2', 'precio_m2_usd', 'pendiente_media_grados', 'score_servicios']].describe().round(2))

## 3. Features seleccionadas y Preprocesamiento anti-leakage

La función `enrich_ml_features()` enriquece cada lote con señales reales sin introducir fuga de datos (*data leakage*), complementada dentro del `Pipeline` de `scikit-learn` por `build_preprocessor()`:

| Bloque de Features | Variables seleccionadas | Qué captura para el precio hedónico |
| :--- | :--- | :--- |
| **Físicas y servicios** | `superficie_m2`, `log_superficie`, `inv_sqrt_sup`, `score_servicios`, `tiene_agua`, `tiene_luz`, `tiene_escritura`, `tiene_arboleda`, `tiene_rio`, `tiene_servicios` | Economías de escala en el precio/m² (lotes más grandes cuestan menos por m²) y prima por infraestructura/escritura. |
| **Topografía y ambiente** | `pendiente_media_grados`, `ndvi_sombra_verano`, `distancia_rio_m` | Costo constructivo por relieve, sombra natural (NDVI MODIS) y cercanía a cursos de agua. |
| **Geoespacial y demanda turística** | `lat_eff`, `lon_eff`, `lat_lon_rot1/2`, `valle_corredor`, `barrio_localidad`, `dist_loc_0..15`, `dist_centroide_min`, `campings_2km/5km/10km`, `distancia_camping_min_km`, `places_5km`, `log_reviews_5km`, `nps_proxy_zona` + `KMeans` geo | Micro-localización continua hacia los 16 centroides urbanos, densidad de competencia turística y demanda medida en Google Maps. |
| **Texto / NLP (avisos reales)** | 10 flags `kw_*` (`kw_lago_dique`, `kw_barrio_privado`, `kw_ruta_asfalto`, `kw_construido`, `kw_financiacion`, `kw_esquina`, `kw_vista`, `kw_loteo`, `kw_gas`, `kw_alambrado`), `txt_len`, `es_tienda_oficial_int` + `TfidfVectorizer` + `TruncatedSVD(20)` | Distingue terrenos frente al lago, en barrios cerrados, sobre ruta asfaltada o con mejoras construidas. |

**Columnas explícitamente excluidas del vector de entrada `X`:**
- **Anti-leakage:** `precio_usd` y `precio_m2_usd` (target derivado).
- **Privacidad (PII):** `telefono_en_descripcion`, `vendedor_nombre`, `url_vendedor` (eliminadas en las salidas).

In [ ]:
df_enriched, feat_meta = enrich_ml_features(df_clean, rules)
print(f'Total features numéricas: {len(feat_meta["num_features"])}')
print(f'Total features categóricas (OneHotEncoder): {len(feat_meta["cat_features"])} -> {feat_meta["cat_features"]}')
print(f'Columna textual (TF-IDF + TruncatedSVD 20 comp.): {feat_meta["text_column"]}')
print(f'Total columnas de entrada X antes de OHE/KMeans/SVD: {len(feat_meta["feature_columns"])}')

## 4. Entrenamiento anti-leakage, comparación de 7 modelos y Métricas ML

Protocolo aplicado en [`src/training.py`](../src/training.py):
1. **Split estricto previo (`80% train` / `20% test`)** estratificado por `valle_corredor` antes de ajustar `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, `KMeans` y `TfidfVectorizer`/`TruncatedSVD`.
2. **Validación cruzada 5-Fold (`KFold`, `shuffle=True`, `random_state=42`)** sobre `train` para comparar 7 candidatos desde un `DummyRegressor` hasta un `VotingRegressor` (`Ensemble 2ET + GBR + HGBR`).
3. **Verificación de serialización Pickle (`.pkl`):** exige `dif_max < 1e-6` entre predicciones en memoria y tras deserializar.

In [ ]:
res = train_and_evaluate_hedonic_model(df_clean, rules)
df_comp = pd.DataFrame(res['comparison'])
display(df_comp)

winner = res['winner_metrics']
dummy = df_comp.loc[df_comp['modelo'].str.contains('Dummy')].iloc[0]
mae_gain_pct = (dummy['mae_test_usd_m2'] - winner['mae_test_usd_m2']) / dummy['mae_test_usd_m2'] * 100.0

print(f"Modelo ganador por CV: {res['bundle']['modelo_nombre']}")
print(f"R2 CV (5-Fold): {winner['r2_cv_mean']:.4f} +/- {winner['r2_cv_std']:.4f}")
print(f"R2 Test (log):  {winner['r2_test_log']:.4f}")
print(f"MAE Test:       {winner['mae_test_usd_m2']:.2f} USD/m2 (reduccion de {mae_gain_pct:.1f}% vs Dummy {dummy['mae_test_usd_m2']:.2f})")
print(f"RMSE Test:      {winner['rmse_test_usd_m2']:.2f} USD/m2")
print(f"dif_max Pickle: {res['dif_max']:.2e} (< 1e-6 requerido)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Top 12 Feature Importances
df_imp = pd.DataFrame(res['feature_importances']).head(12).sort_values('importancia_pct')
df_imp.plot(kind='barh', x='feature', y='importancia_pct', legend=False, ax=axes[0], color='#2a9d8f')
axes[0].set_title('Top 12 Importancias de Variables (%)')
axes[0].set_xlabel('Importancia (%)')

# Panel 2: Precio Real vs Predicho (OOF en Train + Out-of-Sample en Test)
df_sc = res['df_scored']
for split_name, color in [('train_oof', '#264653'), ('test', '#e76f51')]:
    sub = df_sc[df_sc['split_evaluacion'] == split_name]
    axes[1].scatter(sub['precio_m2_usd'], sub['precio_modelo_m2'], alpha=0.5, s=18, label=split_name, color=color)
lim = max(df_sc['precio_m2_usd'].max(), df_sc['precio_modelo_m2'].max())
axes[1].plot([0, lim], [0, lim], '--', color='#555555', linewidth=1.2, label='Identidad y=x')
axes[1].set_title('Precio Real vs. Predicho por Modelo (USD/m2)')
axes[1].set_xlabel('Precio publicado real (USD/m2)')
axes[1].set_ylabel('Precio hedónico modelo (USD/m2)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Métricas de negocio y detección de arbitraje (OOF + Test)

Un lote se clasifica como oportunidad (`es_oportunidad_ml = True`) únicamente si cumple las tres condiciones de negocio simultáneamente:
1. `gap_arbitraje_pct >= 20%` respecto al valor justo predicho `precio_modelo_m2`.
2. `pendiente_media_grados <= 12°`.
3. `riesgo_ambiental != 'ALTO'` (reducción de inviabilidad técnica al `0%`).

Además, se etiqueta `flag_revision = 'REVISAR'` cuando `gap_arbitraje_pct >= 50%` para priorizar auditoría manual ante posibles errores de publicación.

In [ ]:
oport = res['df_oportunidades']
n_revisar = int((oport['flag_revision'] == 'REVISAR').sum()) if len(oport) else 0
n_ok = int((oport['flag_revision'] == 'OK').sum()) if len(oport) else 0

print(f"Lotes evaluados (OOF + Test): {len(res['df_scored'])}")
print(f"Oportunidades ML detectadas:  {len(oport)} ({n_ok} OK | {n_revisar} REVISAR >= 50% gap)")
print(f"Riesgo ALTO en oportunidades: {int((oport['riesgo_ambiental'] == 'ALTO').sum())}")

cols_vista = [
    'id_registro', 'barrio_localidad', 'precio_usd', 'superficie_m2',
    'precio_m2_usd', 'precio_modelo_m2', 'gap_arbitraje_pct',
    'pendiente_media_grados', 'riesgo_ambiental', 'flag_revision', 'split_evaluacion'
]
display(oport[cols_vista].head(15))

## 6. Inferencia en producción (solo inferencia — equivalente al Actor 3 de Apify)

En producción, el Actor [`actors/apify_ml/`](../actors/apify_ml/.actor/actor.json) (ID en Apify Cloud: `9bGh3PnH5lMWm3cjc`) **no entrena**: carga el modelo serializado [`models/terrascout_pipeline.pkl`](../models/terrascout_pipeline.pkl) y ejecuta inferencia pura mediante `execute_ml_actor()` / `score_opportunities()`.

A continuación verificamos la carga desde disco del `.pkl` oficial (con fallback automático al modelo recién entrenado en memoria si la versión de `scikit-learn` en Colab difiere de `1.9.1`) y ejecutamos la inferencia de extremo a extremo igual que en la nube.

In [ ]:
pkl_path = REPO_ROOT / 'models' / 'terrascout_pipeline.pkl'
sdk_pkl_bytes = None
try:
    if not pkl_path.is_file():
        raise FileNotFoundError(f'No existe {pkl_path}')
    print(f'Cargando modelo oficial serializado: {pkl_path} ({pkl_path.stat().st_size / 1024 / 1024:.2f} MB)')
    loaded_bundle = load_model_bundle(str(pkl_path))
except Exception as exc:
    print(f'[Colab / compat] Usando modelo recién entrenado en memoria ({exc})')
    sdk_pkl_bytes = res['pkl_bytes']
    loaded_bundle = res['bundle']

print('Metadatos del bundle .pkl:', {
    'modelo_nombre': loaded_bundle['modelo_nombre'],
    'fecha_entrenamiento': loaded_bundle.get('fecha_entrenamiento'),
    'n_filas_entrenamiento': loaded_bundle.get('n_filas_entrenamiento'),
    'sklearn_version': loaded_bundle.get('sklearn_version'),
    'serialization_dif_max': loaded_bundle.get('serialization_dif_max'),
})

# Simular exactamente la llamada de inferencia pura del Actor Apify (sin reentrenamiento)
inf_inp = {
    'dataset_limpio_url': str(REPO_ROOT / 'data' / 'processed' / 'dataset_limpio.csv'),
    'rules_json_url': str(REPO_ROOT / 'config' / 'business_rules.json'),
}
if sdk_pkl_bytes is None:
    inf_inp['model_pkl_url'] = str(pkl_path)

inf_out = execute_ml_actor(inp=inf_inp, sdk_pkl_bytes=sdk_pkl_bytes)
print('Resumen de inferencia Actor ML:', json.dumps(inf_out['summary'], indent=2, ensure_ascii=False))

cols_inf = [
    'id_registro', 'barrio_localidad', 'precio_m2_usd',
    'precio_modelo_m2', 'precio_modelo_total_usd', 'gap_arbitraje_pct',
    'es_oportunidad_ml', 'flag_revision'
]
display(inf_out['df_oportunidades'][cols_inf].head(10))
print('\n--- Ejemplo de alerta resumida (sin PII) para Telegram / Email ---')
print(format_telegram_summary(inf_out['df_oportunidades'], top_n=3))